In [4]:
# ============================================================
# NLP LAB - INSTALLATION CELL
# ============================================================

!pip install -q gensim pyemd scikit-learn textblob vaderSentiment
!pip install -q spacy
!python -m spacy download en_core_web_sm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.0/126.0 kB 4.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 94.4 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [13]:
# ============================================================
# PROGRAM 23
# IMPLEMENTING ATTENTION MECHANISM
# ============================================================

import numpy as np
import tensorflow as tf

from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input,
    Embedding,
    LSTM,
    Dense,
    Layer
)

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences


# ============================================================
# STEP 1: DATASET
# ============================================================

texts = [
    "I love this movie",
    "This movie is amazing",
    "The movie was excellent",
    "I really enjoyed this film",
    "This is a wonderful movie",

    "I hate this movie",
    "This movie is terrible",
    "The movie was boring",
    "I disliked this film",
    "This movie is horrible"
]

labels = np.array([
    1, 1, 1, 1, 1,
    0, 0, 0, 0, 0
])


# ============================================================
# STEP 2: TOKENIZATION
# ============================================================

vocab_size = 1000

tokenizer = Tokenizer(
    num_words=vocab_size,
    oov_token="<OOV>"
)

tokenizer.fit_on_texts(texts)

sequences = tokenizer.texts_to_sequences(texts)

max_length = 6

X = pad_sequences(
    sequences,
    maxlen=max_length,
    padding="post"
)

print("Input Shape:", X.shape)


# ============================================================
# STEP 3: CUSTOM ATTENTION LAYER
# ============================================================

class AttentionLayer(Layer):

    def __init__(self, **kwargs):
        super(AttentionLayer, self).__init__(**kwargs)

    def build(self, input_shape):

        # input_shape:
        # (batch_size, sequence_length, hidden_size)

        hidden_size = input_shape[-1]

        # Trainable weight matrix
        self.W = self.add_weight(
            name="attention_weight",
            shape=(hidden_size, 1),
            initializer="random_normal",
            trainable=True
        )

        # Trainable bias
        self.b = self.add_weight(
            name="attention_bias",
            shape=(1,),
            initializer="zeros",
            trainable=True
        )

        super(AttentionLayer, self).build(
            input_shape
        )

    def call(self, inputs):

        # ----------------------------------------------------
        # Calculate attention score
        # ----------------------------------------------------

        score = tf.tanh(
            tf.matmul(inputs, self.W) + self.b
        )

        # ----------------------------------------------------
        # Convert scores into attention weights
        # ----------------------------------------------------

        attention_weights = tf.nn.softmax(
            score,
            axis=1
        )

        # ----------------------------------------------------
        # Weighted hidden states
        # ----------------------------------------------------

        weighted_inputs = (
            inputs * attention_weights
        )

        # ----------------------------------------------------
        # Context vector
        # ----------------------------------------------------

        context_vector = tf.reduce_sum(
            weighted_inputs,
            axis=1
        )

        return context_vector


# ============================================================
# STEP 4: BUILD MODEL
# ============================================================

inputs = Input(
    shape=(max_length,)
)

# Embedding layer
embedding = Embedding(
    input_dim=vocab_size,
    output_dim=32
)(inputs)

# LSTM
lstm_output = LSTM(
    64,
    return_sequences=True
)(embedding)

# Attention
attention_output = AttentionLayer()(
    lstm_output
)

# Classification
output = Dense(
    1,
    activation="sigmoid"
)(attention_output)

# Create model
model = Model(
    inputs=inputs,
    outputs=output
)


# ============================================================
# STEP 5: COMPILE
# ============================================================

model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)


# ============================================================
# STEP 6: MODEL SUMMARY
# ============================================================

model.summary()


# ============================================================
# STEP 7: TRAIN
# ============================================================

model.fit(
    X,
    labels,
    epochs=20,
    verbose=1
)


# ============================================================
# STEP 8: TEST
# ============================================================

test_text = [
    "this movie is amazing"
]

test_sequence = tokenizer.texts_to_sequences(
    test_text
)

test_padded = pad_sequences(
    test_sequence,
    maxlen=max_length,
    padding="post"
)

prediction = model.predict(
    test_padded,
    verbose=0
)

probability = prediction[0][0]

print("\n================================")
print("ATTENTION MODEL PREDICTION")
print("================================")

print("Probability:", probability)

if probability >= 0.5:
    print("Prediction: POSITIVE")
else:
    print("Prediction: NEGATIVE")

Input Shape: (10, 6)


Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 6)              │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding_1 (Embedding)         │ (None, 6, 32)          │        32,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 6, 64)          │        24,832 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ attention_layer                 │ (None, 64)             │            65 │
│ (AttentionLayer)                │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 56,962 (222.51 KB)

 Trainable params: 56,962 (222.51 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 7s 7s/step - accuracy: 0.7000 - loss: 0.6926
Epoch 2/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 173ms/step - accuracy: 0.6000 - loss: 0.6923
Epoch 3/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 316ms/step - accuracy: 0.6000 - loss: 0.6918
Epoch 4/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 288ms/step - accuracy: 0.7000 - loss: 0.6914
Epoch 5/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 190ms/step - accuracy: 0.7000 - loss: 0.6909
Epoch 6/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 146ms/step - accuracy: 0.8000 - loss: 0.6904
Epoch 7/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 252ms/step - accuracy: 0.7000 - loss: 0.6898
Epoch 8/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 82ms/step - accuracy: 0.8000 - loss: 0.6892
Epoch 9/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 95ms/step - accuracy: 0.8000 - loss: 0.6885
Epoch 10/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 127ms/step - accuracy: 0.9000 - loss: 0.6877
Epoch 11/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 82ms/step - accuracy: 0.9000 - loss: 0.6868
Epoch 12/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 137ms/step - accuracy: 0.9000 - loss: 0.